In [1]:
# NLG HOSTAGE — Ollama lokal dipaksa memakai CPU, bukan GPU.
# NLG API-first: OpenAI dari .env, Ollama Docker sebagai fallback otomatis.
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "modules").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.hostage_nlg import build_nlg_provider

NLG_PROVIDER = os.getenv("NLG_PROVIDER", "api")
NLG_API_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
NLG_OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen3:8b")
NLG_OLLAMA_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11435")
llm, nlg_provider_config = build_nlg_provider(
    provider=NLG_PROVIDER,
    api_model=NLG_API_MODEL,
    ollama_model=NLG_OLLAMA_MODEL,
    ollama_base_url=NLG_OLLAMA_URL,
)
print(f"NLG HOSTAGE siap | utama={nlg_provider_config['primary_provider']} | fallback={nlg_provider_config['fallback_provider']}")

NLG HOSTAGE siap | utama=openai_api | fallback=ollama_docker


In [2]:
# Pipeline aktif: SVM baseline dan Transformer dibandingkan dengan Fuzzy state yang sama.
from pathlib import Path
import joblib
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "modules").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.hostage_fuzzy import calculate_bluff_indicator, describe_bluff_level
from modules.hostage_nlg import GameMemory, generate_npc_response
from modules.nlu_training import predict_transformer_intent

GAME_MEMORY = GameMemory()
SVM_MODEL = None

def _model_dir():
    for base in (Path.cwd(), PROJECT_ROOT):
        candidate = base / "models"
        if (candidate / "intent_classifier_svm.pkl").is_file():
            return candidate
    raise FileNotFoundError("Model SVM baseline belum tersedia. Jalankan notebook NLU terlebih dahulu.")

def get_svm_intent(user_text):
    global SVM_MODEL
    if SVM_MODEL is None:
        SVM_MODEL = joblib.load(_model_dir() / "intent_classifier_svm.pkl")
    return str(SVM_MODEL.predict([user_text])[0])

def get_transformer_intent(user_text):
    return predict_transformer_intent(user_text, _model_dir(), device="cuda")

def reset_game_memory():
    GAME_MEMORY.entries.clear()

def npc_respond_with_intent(chat_pemain, intent, game_state, npc_name="NPC", speaker_pemain="Pemain"):
    return generate_npc_response(
        llm=llm,
        chat_pemain=chat_pemain,
        intent=intent,
        game_state=game_state,
        npc_name=npc_name,
        memory=GAME_MEMORY,
        speaker_pemain=speaker_pemain,
    )


In [3]:
# Uji langsung: SVM baseline dan Transformer memakai state Fuzzy yang sama.
reset_game_memory()
GAME_MEMORY.add("Dimas", "Naya tadi minta alibi B sebelum Gag Order dipakai.", phase="diskusi", round_number=3, target="Naya")
game_state = {
    "phase": "diskusi",
    "round": 3,
    "accusation_count": 7,
    "claim_contradiction": 8,
    "public_evidence": 7,
    "silence_anomaly": 4,
    "vote_pressure": 6,
    "public_events": ["Gag Order dipakai saat diskusi.", "Tidak ada target Hostage yang diumumkan sistem."],
    "silent_players": ["B"],
}
input_chat = "Si B kena Gag Order pas mau jelasin alibi; menurut gw dia patut dicurigai sebagai Hitman."
bluff_score = calculate_bluff_indicator(7, 8, 7, 4, 6)

svm_intent = get_svm_intent(input_chat)
svm_result = npc_respond_with_intent(input_chat, svm_intent, game_state, npc_name="Naya", speaker_pemain="Raka")
reset_game_memory()
transformer_intent, transformer_confidence = get_transformer_intent(input_chat)
transformer_result = npc_respond_with_intent(input_chat, transformer_intent, game_state, npc_name="Naya", speaker_pemain="Raka")
night_result = npc_respond_with_intent(input_chat, svm_intent, {**game_state, "phase": "malam"}, npc_name="Naya", speaker_pemain="Raka")

print(f"Chat pemain: {input_chat}")
print(f"Fuzzy publik: {bluff_score:.2f}% ({describe_bluff_level(bluff_score).upper()})")
print(f"SVM baseline | intent={svm_intent} | NLG={svm_result['nlg_provider']} | NPC={svm_result['npc_reply']}")
print(f"Transformer  | intent={transformer_intent} ({transformer_confidence:.2f}%) | NLG={transformer_result['nlg_provider']} | NPC={transformer_result['npc_reply']}")
print(f"Fase malam terkunci: {not night_result['response_allowed']} | {night_result['system_message']}")


C:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chat pemain: Si B kena Gag Order pas mau jelasin alibi; menurut gw dia patut dicurigai sebagai Hitman.
Fuzzy publik: 69.93% (SEDANG)
SVM baseline | intent=accusing | NLG=openai_api | NPC=Apa dasar tuduhanmu terhadap B sebagai Hitman? Gag Order bisa jadi alasan dia diam, bukan?
Transformer  | intent=accusing (63.05%) | NLG=openai_api | NPC=Apa dasar kamu mencurigai B sebagai Hitman? Gag Order bisa jadi karena alasan lain, bukan?
Fase malam terkunci: True | Chat terkunci selama fase malam.


## Laporan eksekusi Combine

NLG memakai OpenAI API sebagai provider utama; Ollama Docker tersedia sebagai fallback. Kedua jalur memakai Fuzzy publik yang sama: 69.93% (sedang).

- SVM baseline: accusing. Transformer: accusing (63.05%).
- Kedua respons NPC meminta bukti publik dan tidak membocorkan role, target Hostage, Guard, atau Peek.
- Uji fase malam berhasil: chat terkunci dan NPC tidak mengirim respons.